# Kompleksowa Analiza Statystyczna - Zbiór Danych Detekcji Phishingu


## 1. Import Bibliotek i Wczytanie Danych


In [1]:
from pathlib import Path
import os
import sys
import importlib

start_dir = Path.cwd().resolve()
backend_root = None
for candidate in (start_dir, *start_dir.parents):
    if (candidate / 'utils').is_dir() and (candidate / 'notebooks').is_dir():
        backend_root = candidate
        break
    nested_backend = candidate / 'backend'
    if (nested_backend / 'utils').is_dir() and (nested_backend / 'notebooks').is_dir():
        backend_root = nested_backend
        break
if backend_root is None:
    raise RuntimeError(f'Cannot locate backend root from: {start_dir}')

notebooks_dir = backend_root / 'notebooks'
os.chdir(notebooks_dir)
if str(backend_root) not in sys.path:
    sys.path.insert(0, str(backend_root))

import warnings
import numpy as np
import pandas as pd
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from utils.figure_localization import save_bilingual_plotly_figure
from sklearn.ensemble import RandomForestClassifier

from utils.feature_names import FEATURE_CATEGORIES, get_readable_name
if 'utils.notebook_plot_style' in sys.modules:
    importlib.reload(sys.modules['utils.notebook_plot_style'])

from utils.notebook_plot_style import (
    plotly_tick_format,
    scientific_scale,
    CLASS_COLORS,
)
from utils.split_dataset_loader import (
    METADATA_COLUMNS,
    build_stratified_frames,
    feature_columns,
    load_split_frames,
)

warnings.filterwarnings('ignore')
CORPUS_NAME = 'url_corpus'
TARGET_COL = 'phishing'

if CORPUS_NAME == 'original':
    raw_df = pd.read_csv(backend_root / 'data' / 'dataset_full.csv')
    analysis_frames = build_stratified_frames(raw_df, validation_size=0.2, test_size=0.2)
    output_dir = backend_root.parent / 'results' / 'url_corpus' / 'statistical_analysis'
else:
    split_dir = backend_root / 'data' / 'url_corpus' / 'quality_v4_repaired_features'
    analysis_frames = load_split_frames(split_dir, 'dataset_full')
    output_dir = backend_root.parent / 'results' / 'url_corpus' / 'statistical_analysis'

df = analysis_frames['train'].copy()
analysis_validation_df = analysis_frames['val'].copy()
FEATURE_COLUMNS = feature_columns(df)
csv_dir = output_dir / 'csv'
png_dir = output_dir / 'png'
html_dir = output_dir / 'html'
docs_dir = output_dir / 'docs'
for directory in [output_dir, csv_dir, png_dir, html_dir, docs_dir]:
    directory.mkdir(parents=True, exist_ok=True)

px.defaults.template = 'plotly_white'
px.defaults.color_continuous_scale = px.colors.sequential.Cividis
px.defaults.color_discrete_sequence = [CLASS_COLORS['safe'], CLASS_COLORS['phishing']]

print('✅ Dane załadowane pomyślnie!')
print(f'Korpus: {CORPUS_NAME}')
print(f'Rozmiar zbioru analitycznego: {df.shape}')
print(f'Liczba cech modelowych: {len(FEATURE_COLUMNS)}')
print(f'Testowanie: {len(analysis_validation_df):,} próbek')
print()
print('Kategorie cech:')
for category, features in FEATURE_CATEGORIES.items():
    print(f'  • {category}: {len(features)} cech')
df.head()


✅ Dane załadowane pomyślnie!
Korpus: url_corpus
Rozmiar zbioru analitycznego: (108762, 114)
Liczba cech modelowych: 110
Testowanie: 21,676 próbek

Kategorie cech:
  • URL Properties: 21 cech
  • Domain Properties: 20 cech
  • Directory Properties: 18 cech
  • File Properties: 18 cech
  • Parameter Properties: 22 cech
  • External/DNS Properties: 18 cech


,source_index,url,missing_whois,qty_dot_url,qty_hyphen_url,qty_underline_url,qty_slash_url,qty_questionmark_url,qty_equal_url,qty_at_url,...,qty_dollar_file,tld_present_params,qty_equal_params,qty_exclamation_params,qty_tilde_params,qty_comma_params,qty_plus_params,qty_hashtag_params,qty_dollar_params,phishing
0,16,https://whtsaapp.vip/,1,1,0,0,3,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,11,http://www.holiganbetadresi.com/,1,2,0,0,3,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,35,https://lgtviot.com/pricing,1,1,0,0,3,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,30,https://bafybeihrfdsd7s4oi3tgcj663rsz5etbixwzu...,1,3,0,0,3,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,41,http://43.134.178.58/servicelogin?continue=htt...,1,7,1,1,9,3,6,0,...,0,1,6,0,0,0,0,0,0,1


## 2. Przetwarzanie Wstępne - Czyszczenie i Obsługa Brakujących Danych


In [2]:
def clean_and_analyze_data(df):
    print("=" * 80)
    print("ANALIZA JAKOŚCI DANYCH")
    print("=" * 80)

    print("\n1. PRZEGLĄD ZBIORU DANYCH:")
    print(f"   • Wiersze: {df.shape[0]:,}")
    print(f"   • Kolumny: {df.shape[1]}")
    print(f"   • Cechy: {len(feature_columns(df))}")
    print(f"   • Zmienna docelowa: {TARGET_COL}")

    print("\n2. TYPY DANYCH:")
    dtype_counts = df.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"   • {dtype}: {count} kolumn")

    print("\n3. ANALIZA BRAKUJĄCYCH WARTOŚCI:")
    missing_values = df.isnull().sum()
    if missing_values.sum() == 0:
        print("   ✅ Nie znaleziono brakujących wartości!")
    else:
        missing_percentage = (missing_values / len(df)) * 100
        missing_df = pd.DataFrame({
            'Kolumna': missing_values.index, 'Brakujące': missing_values.values, 'Procent': missing_percentage.values
        })
        print("   ⚠️  Wykryto brakujące wartości:")
        print(missing_df[missing_df['Brakujące'] > 0])

    print("\n4. ANALIZA DUPLIKATÓW:")
    duplicates = df.duplicated().sum()
    print(f"   • Zduplikowane wiersze: {duplicates} ({duplicates/len(df)*100:.2f}%)")

    print("\n5. ROZKŁAD ZMIENNEJ DOCELOWEJ:")
    target_col = TARGET_COL
    target_counts = df[target_col].value_counts().sort_index()

    for value, count in target_counts.items():
        percentage = (count / len(df)) * 100
        label = "Brak phishingu" if value == 0 else "Phishing"
        print(f"   • {label} (klasa {value}): {count:,} próbek ({percentage:.1f}%)")

    class_ratio = target_counts.min() / target_counts.max()
    print(f"\n   Współczynnik balansu klas: {class_ratio:.2f}")
    if class_ratio < 0.5:
        print("   ⚠️  Zbiór jest niezbalansowany")
    else:
        print("   ✅ Zbiór jest rozsądnie zbalansowany")

    df_clean = df.copy()
    if duplicates > 0:
        df_clean = df_clean.drop_duplicates()
        print(f"\n6. AKCJE CZYSZCZENIA:")
        print(f"   ✓ Usunięto {duplicates} zduplikowanych wierszy")

    print("\n" + "=" * 80)
    print("CZYSZCZENIE DANYCH ZAKOŃCZONE")
    print("=" * 80)
    print(f"Końcowy rozmiar zbioru danych: {df_clean.shape}")

    return df_clean

df_clean = clean_and_analyze_data(df)

ANALIZA JAKOŚCI DANYCH

1. PRZEGLĄD ZBIORU DANYCH:
   • Wiersze: 108,762
   • Kolumny: 114
   • Cechy: 110
   • Zmienna docelowa: phishing

2. TYPY DANYCH:
   • int64: 112 kolumn
   • object: 1 kolumn
   • float64: 1 kolumn

3. ANALIZA BRAKUJĄCYCH WARTOŚCI:
   ✅ Nie znaleziono brakujących wartości!

4. ANALIZA DUPLIKATÓW:
   • Zduplikowane wiersze: 0 (0.00%)

5. ROZKŁAD ZMIENNEJ DOCELOWEJ:
   • Brak phishingu (klasa 0): 54,381 próbek (50.0%)
   • Phishing (klasa 1): 54,381 próbek (50.0%)

   Współczynnik balansu klas: 1.00
   ✅ Zbiór jest rozsądnie zbalansowany

CZYSZCZENIE DANYCH ZAKOŃCZONE
Końcowy rozmiar zbioru danych: (108762, 114)


## 3. Statystyki Opisowe


In [3]:
def calculate_descriptive_statistics(df):
    print("=" * 80)
    print("ANALIZA STATYSTYK OPISOWYCH")
    print("=" * 80)

    numeric_columns = [col for col in feature_columns(df) if pd.api.types.is_numeric_dtype(df[col])]
    stats_df = df[numeric_columns].describe().transpose()

    stats_df['zakres'] = stats_df['max'] - stats_df['min']
    stats_df['IQR'] = stats_df['75%'] - stats_df['25%']
    stats_df['skew'] = df[numeric_columns].skew()
    stats_df['kurtosis'] = df[numeric_columns].kurt()

    stats_df = stats_df.rename(columns={
        'mean': 'srednia',
        'std': 'odch_std',
        '25%': 'Q1',
        '50%': 'mediana',
        '75%': 'Q3'
    })

    print("\nPODSTAWOWE STATYSTYKI (Top 20 cech):")
    print("=" * 80)
    display_stats = stats_df[['min', 'max', 'srednia', 'odch_std', 'mediana', 'Q1', 'Q3', 'zakres', 'IQR']].head(20).round(3).copy()
    display_stats.index = [get_readable_name(feat) for feat in display_stats.index]
    print(display_stats)

    print("\nKSZTAŁT ROZKŁADU (Top 20 cech):")
    print("=" * 80)
    shape_stats = stats_df[['skew', 'kurtosis']].head(20).round(3).copy()
    shape_stats.index = [get_readable_name(feat) for feat in shape_stats.index]
    print(shape_stats)

    stats_df.round(4).to_csv(f'{csv_dir}/descriptive_statistics.csv')
    print(f"\n✓ Statystyki zapisane do: {csv_dir}/descriptive_statistics.csv")

    print("\nINTERPRETACJA ROZKŁADU:")
    print("=" * 80)
    highly_skewed = stats_df[abs(stats_df['skew']) > 1].index.tolist()
    highly_kurtotic = stats_df[abs(stats_df['kurtosis']) > 3].index.tolist()
    print(f"   • Silnie skośne cechy (|skośność| > 1): {len(highly_skewed)}")
    if highly_skewed:
        readable_skewed = [get_readable_name(feat) for feat in highly_skewed[:5]]
        print(f"     Przykłady: {', '.join(readable_skewed)}")
    print(f"   • Silnie kurtyczne cechy (|kurtoza| > 3): {len(highly_kurtotic)}")
    if highly_kurtotic:
        readable_kurtotic = [get_readable_name(feat) for feat in highly_kurtotic[:5]]
        print(f"     Przykłady: {', '.join(readable_kurtotic)}")

    return stats_df

descriptive_stats = calculate_descriptive_statistics(df_clean)

ANALIZA STATYSTYK OPISOWYCH



PODSTAWOWE STATYSTYKI (Top 20 cech):
                                min      max  srednia  odch_std  mediana  \
URL: Liczba "." w całym URL     1.0     46.0    1.776     1.367      1.0   
URL: Liczba "-" w całym URL     0.0    100.0    0.824     2.292      0.0   
URL: Liczba "_" w całym URL     0.0     66.0    0.188     1.062      0.0   
URL: Liczba "/" w całym URL     2.0     63.0    3.882     1.669      3.0   
URL: Liczba "?" w całym URL     0.0     26.0    0.095     0.342      0.0   
URL: Liczba "=" w całym URL     0.0    131.0    0.214     1.108      0.0   
URL: Liczba "@" w całym URL     0.0     20.0    0.009     0.122      0.0   
URL: Liczba "&" w całym URL     0.0     52.0    0.126     0.801      0.0   
URL: Długość Top Level Domain   0.0     22.0    1.447     0.931      2.0   
URL: Całkowita długość         13.0  25515.0   52.865   100.356     33.0   
Domena: Liczba "."              0.0      3.0    1.116     0.394      1.0   
Domena: Liczba "-"              0.0     10.0    0.

## 4. Analiza Korelacji


In [4]:
def analyze_correlations(df):
    print("=" * 80)
    print("ANALIZA KORELACJI")
    print("=" * 80)

    target_col = TARGET_COL
    feature_cols = [col for col in feature_columns(df) if pd.api.types.is_numeric_dtype(df[col])]

    print(f"\n1. KORELACJE ZE ZMIENNĄ DOCELOWĄ ({target_col}):")
    target_correlations = df[feature_cols].corrwith(df[target_col]).to_frame('Korelacja')
    target_correlations['Abs_Korelacja'] = target_correlations['Korelacja'].abs()
    target_correlations = target_correlations.sort_values('Abs_Korelacja', ascending=False)

    print(f"\nTop 20 Cech Najbardziej Skorelowanych z '{target_col}':")
    top_20_corr = target_correlations.head(20).copy()
    top_20_corr.index = [get_readable_name(feat) for feat in top_20_corr.index]
    print(top_20_corr.to_string())

    positive = target_correlations[
        target_correlations['Korelacja'] > 0
    ].nlargest(10, 'Korelacja')
    negative = target_correlations[
        target_correlations['Korelacja'] < 0
    ].nsmallest(10, 'Korelacja')
    displayed = pd.concat([negative, positive]).sort_values('Korelacja')
    labels = [get_readable_name(feature) for feature in displayed.index]
    values = displayed['Korelacja'].to_numpy()
    colors = [
        CLASS_COLORS['safe'] if value < 0 else CLASS_COLORS['phishing']
        for value in values
    ]
    value_labels = [f'{value:.2f}'.replace('.', ',') for value in values]
    fig = go.Figure(go.Bar(
        x=values, y=labels, orientation='h',
        marker=dict(color=colors),
        text=value_labels, textposition='outside', cliponaxis=False,
        hovertemplate=(
            '%{y}<br>Współczynnik korelacji Pearsona: '
            '%{x:.3f}<extra></extra>'
        ),
    ))
    lower, upper, ticks = scientific_scale(
        values, symmetric=True, reserve_ratio=0.08,
        target_intervals=4, rounded_bounds=False,
    )
    fig.update_layout(
        width=1400, height=1000,
        xaxis_title='Współczynnik korelacji Pearsona',
        yaxis_title='Cecha', showlegend=False,
        plot_bgcolor='white', separators=', ',
        margin=dict(l=360, r=100, t=40, b=90),
    )
    fig.update_xaxes(
        range=[lower, upper], tickmode='array', tickvals=ticks,
        zeroline=True, zerolinewidth=2,
        zerolinecolor='black', showgrid=True, gridcolor='lightgray',
        tickformat=plotly_tick_format(ticks),
    )
    fig.update_yaxes(showgrid=False)
    save_bilingual_plotly_figure(
        fig, png_dir, html_dir, 'target_correlations',
        scale=2, width=1400, height=1000,
    )

    print("\n2. ANALIZA WIELOKOLINEARNOŚCI:")
    correlation_matrix = df[feature_cols].corr()
    high_corr_pairs = correlation_matrix.unstack().sort_values(ascending=False)
    high_corr_pairs = high_corr_pairs[high_corr_pairs > 0.7]
    high_corr_pairs = high_corr_pairs[high_corr_pairs < 1.0]

    if not high_corr_pairs.empty:
        print(f"   Znaleziono {len(high_corr_pairs)} wysoko skorelowanych par cech (|r| > 0.7)")
        print("\n   Top 10 skorelowanych par:")
        top_10_pairs = high_corr_pairs.head(10).copy()
        top_10_pairs.index = [(get_readable_name(idx[0]), get_readable_name(idx[1])) for idx in top_10_pairs.index]
        print(top_10_pairs)
    else:
        print("   ✅ Nie wykryto wysokiej wielokolinearności (|r| > 0.7)")

    target_correlations.to_csv(f'{csv_dir}/target_correlations.csv')
    print(f"\n✓ Analiza korelacji zapisana do: {csv_dir}/target_correlations.csv")
    print(f"✓ Wizualizacja zapisana do: {png_dir}/target_correlations.png")

    return target_correlations, correlation_matrix

target_correlations, correlation_matrix = analyze_correlations(df_clean)


ANALIZA KORELACJI

1. KORELACJE ZE ZMIENNĄ DOCELOWĄ (phishing):



Top 20 Cech Najbardziej Skorelowanych z 'phishing':
                              Korelacja  Abs_Korelacja
Brak danych DNS                0.507652       0.507652
DNS: Czy ma SPF               -0.478411       0.478411
DNS: Liczba serwerów MX       -0.447420       0.447420
Brak danych TTL                0.397290       0.397290
URL: Liczba "." w całym URL    0.396572       0.396572
HTTP: Liczba przekierowań     -0.393889       0.393889
Brak danych HTTP               0.390290       0.390290
DNS: Liczba serwerów NS       -0.385569       0.385569
Ścieżka: Liczba "/"            0.374346       0.374346
SSL: Czy ma ważny certyfikat  -0.371031       0.371031
Brak danych SSL                0.366703       0.366703
Plik: Liczba "."               0.355582       0.355582
URL: Liczba "/" w całym URL    0.353911       0.353911
DNS: Czas odpowiedzi          -0.328820       0.328820
URL: Liczba "?" w całym URL    0.277316       0.277316
Ścieżka: Długość               0.269120       0.269120
URL: Całkowi


2. ANALIZA WIELOKOLINEARNOŚCI:


   Znaleziono 64 wysoko skorelowanych par cech (|r| > 0.7)

   Top 10 skorelowanych par:
(URL: Liczba "&" w całym URL, Parametry: Liczba "&")     0.983455
(Parametry: Liczba "&", URL: Liczba "&" w całym URL)     0.983455
(URL: Liczba "*" w całym URL, Parametry: Liczba "*")     0.958597
(Parametry: Liczba "*", URL: Liczba "*" w całym URL)     0.958597
(URL: Liczba "/" w całym URL, Ścieżka: Liczba "/")       0.935658
(Ścieżka: Liczba "/", URL: Liczba "/" w całym URL)       0.935658
(Parametry: Liczba parametrów, Parametry: Liczba "=")    0.917170
(Parametry: Liczba "=", Parametry: Liczba parametrów)    0.917170
(Brak danych SSL, Brak danych TTL)                       0.915321
(Brak danych TTL, Brak danych SSL)                       0.915321
dtype: float64

✓ Analiza korelacji zapisana do: /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/results/url_corpus/statistical_analysis/csv/target_correlations.csv
✓ Wizualizacja zapisana do: /Users/maksymiliansowula/Documents/St

## 5. Analiza Ważności Cech (Random Forest)


In [5]:
def calculate_feature_importance(df, validation_df, n_estimators=100):
    print('=' * 80)
    print('ANALIZA WAŻNOŚCI CECH (RANDOM FOREST)')
    print('=' * 80)

    feature_cols = feature_columns(df)
    X_train, y_train = df[feature_cols], df[TARGET_COL]
    X_validation = validation_df[feature_cols]
    y_validation = validation_df[TARGET_COL]

    print()
    print('1. TRENOWANIE RANDOM FOREST:')
    print(f'   • Liczba drzew: {n_estimators}')
    print(f'   • Zbiór treningowy: {len(X_train):,} próbek')
    print(f'   • Zbiór testowy: {len(X_validation):,} próbek')

    rf_model = RandomForestClassifier(
        n_estimators=n_estimators, random_state=42, n_jobs=-1, max_depth=10
    )
    rf_model.fit(X_train, y_train)
    print(f'   • Dokładność treningowa: {rf_model.score(X_train, y_train):.4f}')
    print(f'   • Dokładność testowa: {rf_model.score(X_validation, y_validation):.4f}')

    feature_importance_df = pd.DataFrame({
        'Cecha': feature_cols,
        'Ważność': rf_model.feature_importances_,
    }).sort_values('Ważność', ascending=False)
    displayed = feature_importance_df.head(20).sort_values('Ważność')
    labels = [get_readable_name(feature) for feature in displayed['Cecha']]
    values = displayed['Ważność'].to_numpy()
    value_labels = [f'{value:.3f}'.replace('.', ',') for value in values]
    fig = go.Figure(go.Bar(
        x=values, y=labels, orientation='h',
        marker=dict(color=CLASS_COLORS['safe']),
        text=value_labels, textposition='outside', cliponaxis=False,
        hovertemplate='%{y}<br>Ważność cechy: %{x:.4f}<extra></extra>',
    ))
    lower, upper, ticks = scientific_scale(
        values, include_zero=True, reserve_ratio=0.08,
        target_intervals=5, rounded_bounds=False,
    )
    fig.update_layout(
        width=1400, height=1000,
        xaxis_title='Ważność cechy',
        yaxis_title='Cecha', showlegend=False,
        plot_bgcolor='white', separators=', ',
        margin=dict(l=360, r=100, t=40, b=90),
    )
    fig.update_xaxes(
        range=[lower, upper], tickmode='array', tickvals=ticks,
        showgrid=True, gridcolor='lightgray', zeroline=False,
        tickformat=plotly_tick_format(ticks),
    )
    fig.update_yaxes(showgrid=False)
    save_bilingual_plotly_figure(
        fig, png_dir, html_dir, 'feature_importance',
        scale=2, width=1400, height=1000,
    )
    feature_importance_df.to_csv(csv_dir / 'feature_importance.csv', index=False)
    return feature_importance_df

feature_importance = calculate_feature_importance(df_clean, analysis_validation_df)


ANALIZA WAŻNOŚCI CECH (RANDOM FOREST)

1. TRENOWANIE RANDOM FOREST:
   • Liczba drzew: 100
   • Zbiór treningowy: 108,762 próbek
   • Zbiór testowy: 21,676 próbek


   • Dokładność treningowa: 0.9320
   • Dokładność testowa: 0.9052


## 6. Analiza Rozkładów i Testy Normalności


### Wyjaśnienie wartości -1 w zbiorze danych

Dataset używa **-1 jako wartości oznaczającej "nie występuje w URL"**:

- **Cechy directory/file/params**: -1 = URL nie zawiera ścieżki/pliku/parametrów
- **Cechy DNS/WHOIS**: -1 = informacja niedostępna (timeout, błąd)
- **Cechy binarne**: -1 = nie można ustalić (np. domain_spf, url_google_index)

To **NIE są dane przeskalowane** - to konwencja oznaczająca nieobecność cechy w URL.


In [6]:
def _normality_record(df, column, sample_size):
    '''Return the Shapiro-Wilk result after excluding missing feature values.'''
    values = df[column].replace(-1, np.nan).dropna()
    sample = values.sample(min(len(values), sample_size), random_state=42)
    if len(sample) < 3:
        statistic, p_value = np.nan, np.nan
    else:
        statistic, p_value = stats.shapiro(sample)
    return {
        'Cecha': column,
        'Statystyka Shapiro-Wilk': statistic,
        'wartość-p': p_value,
        'Rozkład Normalny': bool(p_value > 0.05) if pd.notna(p_value) else False,
    }


def _class_feature_values(df, feature):
    '''Return valid feature values for both target classes.'''
    safe = df.loc[df[TARGET_COL] == 0, feature]
    phishing = df.loc[df[TARGET_COL] == 1, feature]
    return safe[safe != -1].dropna().values, phishing[phishing != -1].dropna().values


def _feature_file_stem(position, feature):
    '''Build a stable filesystem-safe plot name.'''
    safe_name = ''.join(character if character.isalnum() else '_' for character in feature)
    return f'{position:02d}_{safe_name.strip("_")}'


def _new_distribution_figure():
    '''Create an empty feature-distribution figure.'''
    return go.Figure()


def _robust_feature_range(safe_values, phishing_values):
    '''Return a robust display range and the combined source values.'''
    combined = np.concatenate([safe_values, phishing_values])
    quantile_lower, upper = np.quantile(combined, [0.01, 0.99])
    lower = 0.0 if combined.min() >= 0 else quantile_lower
    if lower == upper:
        upper = lower + 1.0
    return combined, float(lower), float(upper)


def _discrete_feature_values(values):
    '''Return observed discrete categories when their axis remains readable.'''
    unique = np.unique(values)
    integers = np.isclose(unique, np.round(unique)).all()
    return unique if integers and unique.size <= 8 else None


def _discrete_edges(categories):
    '''Build category boundaries without adding unobserved integer ticks.'''
    if categories.size == 1:
        return np.array([categories[0] - 0.5, categories[0] + 0.5])
    midpoints = (categories[:-1] + categories[1:]) / 2
    first = categories[0] - (midpoints[0] - categories[0])
    last = categories[-1] + (categories[-1] - midpoints[-1])
    return np.concatenate(([first], midpoints, [last]))


def _adaptive_bin_count(values):
    '''Choose a data-dependent readable bin count using Freedman-Diaconis.'''
    recommended = len(np.histogram_bin_edges(values, bins='fd')) - 1
    if recommended <= 1:
        recommended = len(np.histogram_bin_edges(values, bins='sturges')) - 1
    return int(np.clip(recommended, 3, 6))


def _discrete_ticks(categories):
    '''Limit labels while preserving every observed discrete-value bar.'''
    if categories.size <= 6:
        return categories
    indices = np.linspace(0, categories.size - 1, 5).round().astype(int)
    return categories[np.unique(indices)]


def _integer_interval_edges(categories):
    '''Return full integer boundaries for consecutive count values.'''
    if not np.isclose(np.diff(categories), 1.0).all():
        return None
    return np.arange(categories[0], categories[-1] + 2.0)


def _continuous_geometry(visible, lower, upper):
    '''Return adaptive continuous bins, centres, limits and ticks.'''
    count = _adaptive_bin_count(visible)
    scale = scientific_scale(
        [lower, upper], include_zero=lower >= 0.0, reserve_ratio=0.0,
        target_intervals=count, rounded_bounds=True,
    )
    edges = scale[2]
    centres = (edges[:-1] + edges[1:]) / 2
    return edges, centres, scale


def _distribution_geometry(combined, lower, upper):
    '''Build an axis individually for the observed feature distribution.'''
    visible = combined[(combined >= lower) & (combined <= upper)]
    categories = _discrete_feature_values(visible)
    if categories is None:
        return _continuous_geometry(visible, lower, upper)
    integer_edges = _integer_interval_edges(categories)
    if integer_edges is not None:
        centres = (integer_edges[:-1] + integer_edges[1:]) / 2
        scale = (float(integer_edges[0]), float(integer_edges[-1]), integer_edges)
        return integer_edges, centres, scale
    edges = _discrete_edges(categories)
    scale = (float(edges[0]), float(edges[-1]), _discrete_ticks(categories))
    return edges, categories, scale


def _visible_values(values, lower, upper):
    '''Select observations visible in a robust histogram range.'''
    return values[(values >= lower) & (values <= upper)]


def _histogram_percentages(values, edges):
    '''Calculate percentages that sum to exactly 100.00 in the visible range.'''
    counts, _ = np.histogram(values, bins=edges)
    if counts.sum() == 0:
        return np.zeros(len(edges) - 1)
    raw_units = counts / counts.sum() * 10000
    units = np.floor(raw_units).astype(int)
    remainder = 10000 - int(units.sum())
    order = np.argsort(-(raw_units - units), kind='stable')
    units[order[:remainder]] += 1
    return units / 100


def _add_class_histograms(figure, safe_values, phishing_values, edges):
    '''Add grouped class histograms using the same numeric intervals.'''
    safe_percent = _histogram_percentages(safe_values, edges)
    phishing_percent = _histogram_percentages(phishing_values, edges)
    _add_histogram_bars(
        figure, edges, safe_percent,
        'Brak phishingu', CLASS_COLORS['safe'], -1,
    )
    _add_histogram_bars(
        figure, edges, phishing_percent,
        'Phishing', CLASS_COLORS['phishing'], 1,
    )
    safe_visible = _visible_values(safe_values, edges[0], edges[-1])
    phishing_visible = _visible_values(phishing_values, edges[0], edges[-1])
    excluded = len(safe_values) + len(phishing_values)
    excluded -= len(safe_visible) + len(phishing_visible)
    maximum = max(safe_percent.max(), phishing_percent.max(), 0.0)
    return excluded, float(maximum)


def _percentage_labels(percentages):
    '''Format non-zero percentages with consistent Polish precision.'''
    return [
        '' if value == 0 else f'{value:.2f}%'.replace('.', ',')
        for value in percentages
    ]


def _add_histogram_bars(figure, edges, percentages, label, color, side):
    '''Place one class in its half of every shared histogram interval.'''
    widths = np.diff(edges)
    centres = (edges[:-1] + edges[1:]) / 2
    positions = centres + side * widths * 0.25
    intervals = np.column_stack((edges[:-1], edges[1:]))
    figure.add_bar(
        x=positions, y=percentages, width=widths * 0.5, name=label,
        marker=dict(color=color, line=dict(color='black', width=0.8)),
        customdata=intervals, text=_percentage_labels(percentages),
        textposition='outside', cliponaxis=False,
        hovertemplate=(
            'Przedział: [%{customdata[0]:.2~f}, '
            '%{customdata[1]:.2~f})<br>'
            'Udział próbek klasy: '
            '%{y:.2f}%<extra></extra>'
        ),
    )


def _style_distribution_figure(figure, feature_label, y_title, barmode):
    '''Apply a readable scientific layout to one distribution figure.'''
    figure.update_layout(
        width=1200, height=700, barmode=barmode,
        xaxis_title=f'Wartość cechy: "{feature_label}"', yaxis_title=y_title,
        plot_bgcolor='white', font=dict(size=16),
        legend=dict(
            orientation='h', x=0.5, xanchor='center',
            y=1.0, yanchor='bottom',
        ),
        margin=dict(l=110, r=60, t=120, b=110),
        bargap=0.0, bargroupgap=0.0, separators=', ',
    )
    figure.update_xaxes(
        showgrid=True, gridcolor='lightgray', zeroline=False,
        automargin=True, showline=True,
        linecolor='black', linewidth=3, ticks='outside',
        tickcolor='black', tickwidth=2, ticklabeloverflow='allow',
        ticklabelstandoff=8,
    )
    figure.update_yaxes(
        showgrid=True, gridcolor='lightgray',
        rangemode='tozero', tickformat='.1f', automargin=True,
        showline=True, linecolor='black', linewidth=3,
        ticks='outside', tickcolor='black', tickwidth=2,
    )


def _save_distribution_figure(figure, position, feature, png_output, html_output):
    '''Save one distribution as independent raster and interactive files.'''
    stem = _feature_file_stem(position, feature)
    save_bilingual_plotly_figure(
        figure, png_output, html_output, stem,
        scale=2, width=1200, height=700,
    )
    return stem


def _remove_legacy_distribution_files():
    '''Remove obsolete files containing eight distributions in one image.'''
    legacy_files = (
        Path(png_dir) / 'distributions_top8_features.png',
        Path(html_dir) / 'distributions_top8_features.html',
    )
    for path in legacy_files:
        if path.exists():
            path.unlink()


def _plot_feature_distribution(df, feature, position, png_output, html_output):
    '''Create and save one independently scaled feature-distribution chart.'''
    safe_values, phishing_values = _class_feature_values(df, feature)
    figure = _new_distribution_figure()
    feature_label = get_readable_name(feature)
    combined, lower, upper = _robust_feature_range(safe_values, phishing_values)
    edges, _, x_scale = _distribution_geometry(combined, lower, upper)
    excluded, maximum = _add_class_histograms(
        figure, safe_values, phishing_values, edges,
    )
    _style_distribution_figure(
        figure, feature_label,
        'Udział próbek klasy w pokazanym zakresie [%]', 'overlay',
    )
    x_lower, x_upper, x_ticks = x_scale
    y_lower, y_upper, y_ticks = scientific_scale(
        [0.0, max(maximum, 1.0)], include_zero=True, reserve_ratio=0.16,
        target_intervals=5, rounded_bounds=False,
    )
    figure.update_xaxes(
        range=[x_lower, x_upper], tickmode='array',
        tickvals=x_ticks, tickformat=plotly_tick_format(x_ticks),
    )
    figure.update_yaxes(
        range=[y_lower, y_upper], tickmode='array',
        tickvals=y_ticks, tickformat='.2~f',
    )
    total = len(safe_values) + len(phishing_values)
    excluded_percent = excluded / total * 100 if total else 0.0
    print(
        f'{feature_label}: pokazany zakres {edges[0]:.2f}–{edges[-1]:.2f}; '
        f'poza zakresem {excluded} próbek ({excluded_percent:.2f}%).'
    )
    return _save_distribution_figure(
        figure, position, feature, png_output, html_output,
    )


def _missing_value_frame(df, features):
    '''Build class-wise percentages of the -1 missing-value sentinel.'''
    rows = []
    for feature in features:
        safe = df.loc[df[TARGET_COL] == 0, feature]
        phishing = df.loc[df[TARGET_COL] == 1, feature]
        rows.append({
            'Cecha': get_readable_name(feature),
            'Brak phishingu': float((safe == -1).mean() * 100),
            'Phishing': float((phishing == -1).mean() * 100),
            'Brak phishingu (liczba)': int((safe == -1).sum()),
            'Phishing (liczba)': int((phishing == -1).sum()),
        })
    return pd.DataFrame(rows)


def _add_missing_value_bars(figure, missing_df):
    '''Add both target classes to the missing-value comparison chart.'''
    for label, color in (
        ('Brak phishingu', CLASS_COLORS['safe']),
        ('Phishing', CLASS_COLORS['phishing']),
    ):
        values = missing_df[label]
        value_labels = [
            f'{value:.2f}%'.replace('.', ',') if value > 0 else ''
            for value in values
        ]
        figure.add_bar(
            name=label, y=missing_df['Cecha'], x=values,
            orientation='h', marker_color=color,
            text=value_labels, textposition='outside', cliponaxis=False,
            customdata=missing_df[f'{label} (liczba)'],
            hovertemplate=(
                '%{y}<br>Udział wartości -1: %{x:.2f}%'
                '<br>Liczba próbek: %{customdata}<extra></extra>'
            ),
        )


def _plot_missing_values(df, features):
    '''Plot the sentinel-value rate independently of distributions.'''
    missing_df = _missing_value_frame(df, features)
    missing_df.to_csv(Path(csv_dir) / 'missing_values_minus1.csv', index=False)
    missing_df = missing_df[
        missing_df[['Brak phishingu', 'Phishing']].max(axis=1) > 0
    ]
    if missing_df.empty:
        print('Brak wartości -1 w analizowanych cechach.')
        return
    figure = go.Figure()
    _add_missing_value_bars(figure, missing_df)
    maximum = float(missing_df[['Brak phishingu', 'Phishing']].max().max())
    lower, upper, ticks = scientific_scale(
        [0.0, maximum], include_zero=True, reserve_ratio=0.12,
        target_intervals=5, rounded_bounds=False,
    )
    height = max(500, 105 * len(missing_df) + 180)
    figure.update_layout(
        width=1200, height=height, barmode='group',
        xaxis_title='Udział próbek z wartością -1 [%]',
        plot_bgcolor='white', font=dict(size=16),
        legend=dict(
            orientation='h', x=0.5, xanchor='center',
            y=1.08, yanchor='bottom',
        ),
        margin=dict(l=360, r=100, t=80, b=100), separators=', ',
    )
    figure.update_xaxes(
        range=[lower, upper], tickmode='array', tickvals=ticks,
        showgrid=True, gridcolor='lightgray',
        tickformat=plotly_tick_format(ticks),
    )
    figure.update_yaxes(showgrid=False, autorange='reversed')
    save_bilingual_plotly_figure(
        figure, png_dir, html_dir, 'missing_values_minus1',
        scale=2, width=1200, height=height,
    )


def analyze_distributions(df, sample_size=1000):
    '''Analyze normality and save eight independent feature distributions.'''
    print('=' * 80)
    print('ANALIZA ROZKŁADÓW')
    print('=' * 80)
    numeric = [
        column for column in feature_columns(df)
        if pd.api.types.is_numeric_dtype(df[column])
    ]
    normality_df = pd.DataFrame([
        _normality_record(df, column, sample_size) for column in numeric
    ])
    normal_count = int(normality_df['Rozkład Normalny'].sum())
    print(
        f'Cechy z rozkładem normalnym: {normal_count}/{len(normality_df)} '
        '(test Shapiro-Wilka, poziom istotności 0,05)'
    )
    top_features = feature_importance['Cecha'].head(8).tolist()
    _remove_legacy_distribution_files()
    distribution_png = Path(png_dir) / 'distributions'
    distribution_html = Path(html_dir) / 'distributions'
    distribution_png.mkdir(parents=True, exist_ok=True)
    distribution_html.mkdir(parents=True, exist_ok=True)
    for position, feature in enumerate(top_features, start=1):
        _plot_feature_distribution(
            df, feature, position, distribution_png, distribution_html,
        )
    _plot_missing_values(df, top_features)
    normality_df.to_csv(Path(csv_dir) / 'distribution_analysis.csv', index=False)
    print(f'✓ Osobne wykresy rozkładów zapisano w: {distribution_png}')
    print(f'✓ Analizę normalności zapisano w: {csv_dir}/distribution_analysis.csv')
    return normality_df


distribution_analysis = analyze_distributions(df_clean)


ANALIZA ROZKŁADÓW
Cechy z rozkładem normalnym: 36/110 (test Shapiro-Wilka, poziom istotności 0,05)
URL: Liczba "." w całym URL: pokazany zakres 1.00–7.00; poza zakresem 616 próbek (0.57%).


URL: Całkowita długość: pokazany zakres 0.00–300.00; poza zakresem 923 próbek (0.85%).


DNS: Liczba rozwiązanych IP: pokazany zakres 0.00–7.00; poza zakresem 493 próbek (0.61%).


DNS: Liczba serwerów NS: pokazany zakres 0.00–8.00; poza zakresem 354 próbek (0.44%).


DNS: Liczba serwerów MX: pokazany zakres 0.00–8.00; poza zakresem 111 próbek (0.14%).


Plik: Długość nazwy: pokazany zakres 0.00–80.00; poza zakresem 730 próbek (0.67%).


DNS: Czy ma SPF: pokazany zakres 0.00–2.00; poza zakresem 0 próbek (0.00%).


Ścieżka: Długość: pokazany zakres 0.00–120.00; poza zakresem 428 próbek (0.39%).


✓ Osobne wykresy rozkładów zapisano w: /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/results/url_corpus/statistical_analysis/png/distributions
✓ Analizę normalności zapisano w: /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/results/url_corpus/statistical_analysis/csv/distribution_analysis.csv


## 7. Wizualizacja - Mapa Ciepła Korelacji


In [7]:
from textwrap import wrap


def _wrapped_correlation_label(label, width=24):
    '''Wrap a feature label without abbreviating its meaning.'''
    return '<br>'.join(wrap(label, width=width))


def _correlation_labels(columns, target_column):
    '''Return readable, wrapped labels for heatmap axes.'''
    return [
        _wrapped_correlation_label(
            'Klasa docelowa' if column == target_column else get_readable_name(column)
        )
        for column in columns
    ]


def _balanced_correlation_features(top_n):
    '''Select the strongest positive and negative target correlations.'''
    correlations = target_correlations['Korelacja']
    positive_count = top_n // 2
    positive = correlations[correlations > 0].nlargest(positive_count)
    negative = correlations[correlations < 0].nsmallest(top_n - positive_count)
    selected = list(positive.index) + list(negative.index)
    if len(selected) < top_n:
        remaining = correlations.drop(index=selected).abs().nlargest(top_n - len(selected))
        selected.extend(remaining.index.tolist())
    return selected


def _add_correlation_values(figure, matrix, labels):
    '''Annotate every matrix cell using contrast-aware text colors.'''
    for row, column in np.ndindex(matrix.shape):
        value = float(matrix.iloc[row, column])
        label = f'{value:.2f}'.replace('.', ',')
        figure.add_annotation(
            x=labels[column], y=labels[row], text=label,
            showarrow=False,
            font=dict(size=13, color='white' if abs(value) >= 0.55 else 'black'),
        )


def _style_correlation_heatmap(figure):
    '''Apply a large, publication-oriented heatmap layout.'''
    figure.update_layout(
        width=1600, height=1500,
        margin=dict(l=310, r=170, t=50, b=330),
        xaxis=dict(title='Cecha', tickangle=-55, tickfont=dict(size=13)),
        yaxis=dict(title='Cecha', autorange='reversed', tickfont=dict(size=13)),
        plot_bgcolor='white', separators=', ',
    )


def create_correlation_heatmap(df, top_n=10):
    '''Create one readable, complete Pearson correlation matrix.'''
    target_column = TARGET_COL
    top_features = _balanced_correlation_features(top_n)
    matrix = df[top_features + [target_column]].corr()
    labels = _correlation_labels(matrix.columns, target_column)
    figure = go.Figure(data=go.Heatmap(
        z=matrix.values, x=labels, y=labels,
        colorscale=[
            [0.00, '#2F5F8F'], [0.25, '#A9C2D8'], [0.50, '#f7f7f7'],
            [0.75, '#F2B38C'], [1.00, '#B84E16'],
        ],
        zmin=-1, zmax=1, zmid=0, xgap=1, ygap=1,
        colorbar=dict(
            title='Współczynnik<br>korelacji Pearsona',
            tickvals=[-1, -0.5, 0, 0.5, 1],
            ticktext=['-1', '-0,5', '0', '0,5', '1'],
        ),
        hovertemplate=(
            '%{y}<br>%{x}<br>Współczynnik korelacji: '
            '%{z:.3f}<extra></extra>'
        ),
    ))
    _add_correlation_values(figure, matrix, labels)
    _style_correlation_heatmap(figure)
    save_bilingual_plotly_figure(
        figure, png_dir, html_dir, 'correlation_heatmap',
        scale=2, width=1600, height=1500,
    )
    print(f'✓ Mapa korelacji zapisana do: {png_dir}/correlation_heatmap.png')
    return matrix


correlation_heatmap = create_correlation_heatmap(df_clean, top_n=10)


✓ Mapa korelacji zapisana do: /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/results/url_corpus/statistical_analysis/png/correlation_heatmap.png


## 8. Zaawansowana Analiza Selekcji Cech


In [8]:
def advanced_feature_selection_analysis(df, feature_importance_df, correlation_matrix, target_corr_df):
    print("=" * 80)
    print("ZAAWANSOWANA ANALIZA SELEKCJI CECH")
    print("=" * 80)

    feature_cols = feature_columns(df)
    features_to_eliminate = {}

    zero_variance = df[feature_cols].var()[df[feature_cols].var() == 0].index.tolist()
    near_zero_variance = df[feature_cols].var()[(df[feature_cols].var() > 0) & (df[feature_cols].var() < 0.01)].index.tolist()

    features_to_eliminate['zero_variance'] = zero_variance
    features_to_eliminate['near_zero_variance'] = near_zero_variance

    print(f"\n1. WARIANCJA:")
    print(f"   • Cechy o zerowej wariancji: {len(zero_variance)}")
    print(f"   • Cechy o niemal zerowej wariancji (<0.01): {len(near_zero_variance)}")

    corr_threshold = 0.95
    high_corr_pairs = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            if abs(correlation_matrix.iloc[i, j]) > corr_threshold:
                feature_i = correlation_matrix.columns[i]
                feature_j = correlation_matrix.columns[j]

                importance_i = feature_importance_df[feature_importance_df['Cecha'] == feature_i]['Ważność'].values[0] if feature_i in feature_importance_df['Cecha'].values else 0
                importance_j = feature_importance_df[feature_importance_df['Cecha'] == feature_j]['Ważność'].values[0] if feature_j in feature_importance_df['Cecha'].values else 0

                eliminate_feature = feature_i if importance_i < importance_j else feature_j
                high_corr_pairs.append((feature_i, feature_j, correlation_matrix.iloc[i, j], eliminate_feature))

    highly_correlated = list(set([pair[3] for pair in high_corr_pairs]))
    features_to_eliminate['high_correlation'] = highly_correlated

    print(f"\n2. WYSOKA KORELACJA (>{corr_threshold}):")
    print(f"   • Znaleziono par silnie skorelowanych: {len(high_corr_pairs)}")
    print(f"   • Cechy do eliminacji (niższa ważność): {len(highly_correlated)}")

    low_importance = feature_importance_df[feature_importance_df['Ważność'] < 0.001]['Cecha'].tolist()
    features_to_eliminate['low_importance'] = low_importance

    print(f"\n3. WAŻNOŚĆ CECH (Random Forest):")
    print(f"   • Cechy o bardzo niskiej ważności (<0.001): {len(low_importance)}")

    weak_target_corr = target_corr_df[target_corr_df['Abs_Korelacja'] < 0.05].index.tolist()
    features_to_eliminate['weak_target_correlation'] = weak_target_corr

    print(f"\n4. KORELACJA Z ZMIENNĄ DOCELOWĄ:")
    print(f"   • Cechy o słabej korelacji z 'phishing' (<0.05): {len(weak_target_corr)}")

    skewness = df[feature_cols].skew()
    high_skewness = skewness[abs(skewness) > 5].index.tolist()

    kurtosis = df[feature_cols].kurtosis()
    high_kurtosis = kurtosis[abs(kurtosis) > 10].index.tolist()

    print(f"\n5. ROZKŁADY:")
    print(f"   • Cechy o wysokiej skośności (|skewness| > 5): {len(high_skewness)}")
    print(f"   • Cechy o wysokiej kurtozie (|kurtosis| > 10): {len(high_kurtosis)}")

    all_eliminate = list(set(
        zero_variance + near_zero_variance + highly_correlated +
        low_importance + weak_target_corr
    ))

    elimination_counts = {feat: sum([1 for cat_feats in features_to_eliminate.values() if feat in cat_feats])
                         for feat in all_eliminate}

    high_confidence_eliminate = [feat for feat, count in elimination_counts.items() if count >= 2]

    print(f"\n6. PODSUMOWANIE ELIMINACJI:")
    print("=" * 80)
    print(f"   • Łączna liczba unikalnych cech do potencjalnej eliminacji: {len(all_eliminate)}")
    print(f"   • Cechy do eliminacji (wysoka pewność - ≥2 powody): {len(high_confidence_eliminate)}")
    print(f"   • Cechy do przeglądu (średnia pewność - 1 powód): {len(all_eliminate) - len(high_confidence_eliminate)}")

    high_importance = feature_importance_df.nlargest(30, 'Ważność')['Cecha'].tolist()
    strong_correlation = target_corr_df[target_corr_df['Abs_Korelacja'] >= 0.3].index.tolist()

    strong_features = set(high_importance) & set(strong_correlation)
    strong_features = [f for f in strong_features if f not in all_eliminate]

    print(f"   • Cechy z wysoką ważnością I silną korelacją: {len(strong_features)}")
    if strong_features:
        readable_strong = [get_readable_name(f) for f in list(strong_features)[:20]]
        print(f"     Top cechy: {', '.join(readable_strong)}")

    print("\n7. KOŃCOWE REKOMENDACJE SELEKCJI CECH:")
    print("=" * 80)

    features_to_keep_final = [f for f in feature_cols if f not in high_confidence_eliminate]

    print(f"   • Oryginalna liczba cech: {len(feature_cols)}")
    print(f"   • Rekomendowane do ELIMINACJI (wysoka pewność): {len(high_confidence_eliminate)}")
    print(f"   • Rekomendowane do ZACHOWANIA: {len(features_to_keep_final)}")
    print(f"   • Redukcja cech: {len(high_confidence_eliminate)/len(feature_cols)*100:.1f}%")

    elimination_df = pd.DataFrame([
        {'Cecha': feat, 'Pewność_Eliminacji': 'Wysoka' if feat in high_confidence_eliminate else 'Średnia',
         'Liczba_Powodów': elimination_counts.get(feat, 0),
         'Powody': ', '.join([cat for cat, feats in features_to_eliminate.items() if feat in feats])}
        for feat in all_eliminate
    ]).sort_values('Liczba_Powodów', ascending=False)

    elimination_df.to_csv(f'{csv_dir}/features_to_eliminate.csv', index=False)
    print(f"\n✓ Rekomendacje eliminacji zapisane do: {csv_dir}/features_to_eliminate.csv")

    keep_df = pd.DataFrame({
        'Cecha': features_to_keep_final,
        'Ważność_RF': [feature_importance_df[feature_importance_df['Cecha'] == f]['Ważność'].values[0]
                          if f in feature_importance_df['Cecha'].values else 0
                          for f in features_to_keep_final],
        'Korelacja_z_Celem': [target_corr_df.loc[f, 'Korelacja']
                               if f in target_corr_df.index else 0
                               for f in features_to_keep_final]
    }).sort_values('Ważność_RF', ascending=False)

    keep_df.to_csv(f'{csv_dir}/features_to_keep.csv', index=False)
    print(f"✓ Cechy do zachowania zapisane do: {csv_dir}/features_to_keep.csv")

    return elimination_df, keep_df, high_confidence_eliminate, features_to_keep_final

elimination_recommendations, features_to_keep_df, eliminate_list, keep_list = advanced_feature_selection_analysis(
    df_clean, feature_importance, correlation_matrix, target_correlations
)

ZAAWANSOWANA ANALIZA SELEKCJI CECH

1. WARIANCJA:
   • Cechy o zerowej wariancji: 25
   • Cechy o niemal zerowej wariancji (<0.01): 26

2. WYSOKA KORELACJA (>0.95):
   • Znaleziono par silnie skorelowanych: 3
   • Cechy do eliminacji (niższa ważność): 3

3. WAŻNOŚĆ CECH (Random Forest):
   • Cechy o bardzo niskiej ważności (<0.001): 73

4. KORELACJA Z ZMIENNĄ DOCELOWĄ:
   • Cechy o słabej korelacji z 'phishing' (<0.05): 36

5. ROZKŁADY:
   • Cechy o wysokiej skośności (|skewness| > 5): 65
   • Cechy o wysokiej kurtozie (|kurtosis| > 10): 73

6. PODSUMOWANIE ELIMINACJI:
   • Łączna liczba unikalnych cech do potencjalnej eliminacji: 77
   • Cechy do eliminacji (wysoka pewność - ≥2 powody): 63
   • Cechy do przeglądu (średnia pewność - 1 powód): 14
   • Cechy z wysoką ważnością I silną korelacją: 13
     Top cechy: URL: Liczba "." w całym URL, Plik: Liczba ".", Brak danych HTTP, DNS: Czy ma SPF, Brak danych DNS, HTTP: Liczba przekierowań, DNS: Liczba serwerów NS, Brak danych SSL, DNS: Cza

## 9. Tworzenie Zoptymalizowanego Zbioru Danych i Podsumowanie


In [9]:
def create_optimized_dataset_and_summarize(df, keep_list, eliminate_list):
    print("=" * 80)
    print("TWORZENIE ZOPTYMALIZOWANEGO ZBIORU DANYCH I PODSUMOWANIE")
    print("=" * 80)

    target_col = TARGET_COL

    export_df = raw_df if CORPUS_NAME == 'original' else df
    df_optimized = export_df[keep_list + [target_col]].copy()

    print(f"\n1. ZBIÓR DANYCH - SELEKCJA CECH (Feature Selection):")
    print(f"   • Oryginalna liczba cech: {len(feature_columns(df))}")
    print(f"   • Cechy po eliminacji: {len(keep_list)}")
    print(f"   • Usunięte cechy: {len(eliminate_list)}")
    print(f"   • Redukcja: {len(eliminate_list) / (len(feature_columns(df))) * 100:.1f}%")

    if CORPUS_NAME == 'original':
        optimized_path = backend_root / 'data' / 'dataset_optimized.csv'
        df_optimized.to_csv(optimized_path, index=False)
        print(f'   ✓ Zapisano do: {optimized_path}')
    else:
        split_dir = backend_root / 'data' / 'url_corpus' / 'quality_v4_repaired_features'
        for split_name, split_df in analysis_frames.items():
            metadata = [col for col in split_df.columns if col in METADATA_COLUMNS]
            optimized_split = split_df[metadata + keep_list + [TARGET_COL]].copy()
            optimized_path = split_dir / f'dataset_optimized_{split_name}.csv'
            optimized_split.to_csv(optimized_path, index=False)
            print(f'   ✓ Zapisano do: {optimized_path}')

    print(f"\n" + "=" * 80)
    print("PORÓWNANIE ZBIORÓW DANYCH")
    print("=" * 80)

    comparison_data = {
        'Zbiór danych': [
            'Full (pełny)',
            'Optimized (eliminacja cech)'
        ],
        'Liczba cech': [
            len(feature_columns(df)),
            len(keep_list)
        ],
        'Typ cech': [
            'Wszystkie atrybuty (URL + DNS/WHOIS)',
            'Wszystkie atrybuty (URL + DNS/WHOIS)'
        ],
        'Interpretowalność': [
            'Wysoka',
            'Wysoka'
        ],
        'Plik': [
            'dataset_full.csv',
            'dataset_optimized.csv'
        ]
    }

    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))

    summary = f"""
# Podsumowanie Kompleksowej Analizy Statystycznej

## 1. Przegląd Danych
- **Liczba próbek:** {len(df):,}
- **Liczba cech (oryginalnie):** {len(feature_columns(df))}
- **Typ cech:** URL + DNS/WHOIS (wszystkie reprodukowalne)
- **Rozkład klas:**
    - Phishing: {df[target_col].value_counts()[1]:,} ({df[target_col].value_counts(normalize=True)[1]:.2%})
    - Brak phishingu: {df[target_col].value_counts()[0]:,} ({df[target_col].value_counts(normalize=True)[0]:.2%})

## 2. Utworzony Zbiór Danych

### Optimized (Selekcja Cech)
- **Plik:** `dataset_optimized.csv`
- **Liczba cech:** {len(keep_list)} (z {len(feature_columns(df))})
- **Redukcja:** {len(eliminate_list) / (len(feature_columns(df))):.1%}
- **Typ:** Wszystkie atrybuty (URL + DNS/WHOIS)
- **Zalety:**
  - Zachowuje oryginalne cechy - łatwo zrozumieć znaczenie
  - Dobra do analizy biznesowej i wyjaśniania modelu
  - Zawiera cechy z URLFeatureAnalyzer (szybkie) + AdvancedURLAnalyzer (wolne)
  - Najwyższa dokładność modeli
- **Zastosowanie:** Modele wymagające maksymalnej dokładności

## 3. Kluczowe Cechy (Feature Selection)
**Top 5 najważniejszych zachowanych cech:**
"""
    for i, feature in enumerate(feature_importance['Cecha'].head(5), 1):
        importance = feature_importance[feature_importance['Cecha'] == feature]['Ważność'].values[0]
        summary += f"{i}. `{feature}` (ważność: {importance:.4f})\n"

    summary += f"""
## 4. Rekomendacje

### Wybór Zbioru Danych:
- **Dataset Full** → Maksymalna informacja (liczba cech wyliczana z danych)
- **Dataset Optimized** → Najlepszy stosunek jakości do wydajności ({len(keep_list)} cech)

### Zalecane Modele:
- Random Forest, XGBoost, Logistic Regression (dobre do interpretacji)
- SVM, Neural Networks (dla lepszej dokładności)

## 5. Informacja o Reprodukowalności Cech
Wszystkie cechy są reprodukowalne przy użyciu:
- **URLFeatureAnalyzer** - szybka ekstrakcja z URL (89 cech URL-only)
- **AdvancedURLAnalyzer** - wolna ekstrakcja z DNS/WHOIS/HTTP (11 cech):
  - time_response, domain_spf, asn_ip
  - time_domain_activation, time_domain_expiration
  - qty_ip_resolved, qty_nameservers, qty_mx_servers
  - ttl_hostname, url_google_index, domain_google_index

## 6. Następne Kroki
1. Przetestować oba zbiory danych z różnymi modelami
2. Porównać dokładność i czas trenowania
3. Wybrać najlepsze podejście dla aplikacji

## 7. Zapisane Artefakty
- **Dane oryginalne:** `../data/url_corpus/quality_v4_repaired_features/dataset_full_train.csv` ({len(feature_columns(df))} cech)
- **Dane zoptymalizowane:** `../data/url_corpus/quality_v4_repaired_features/dataset_optimized_train.csv` ({len(keep_list)} cech)
- **Wizualizacje PNG:** `../results/original/statistical_analysis/png/`
- **Wizualizacje HTML:** `../results/original/statistical_analysis/html/`
- **Raporty CSV:** `../results/original/statistical_analysis/csv/`
- **Dokumentacja:** `../results/original/statistical_analysis/docs/`
"""

    summary_path = f'{docs_dir}/analysis_summary.md'
    with open(summary_path, 'w') as f:
        f.write(summary)

    print(f"\n2. PODSUMOWANIE ANALIZY ZAPISANE:")
    print(f"   ✓ Lokalizacja: {summary_path}")

    print(f"\n" + "=" * 80)
    print("REKOMENDACJE UŻYCIA:")
    print("=" * 80)
    print("• dataset_full.csv → Maksymalna informacja, liczba cech wyliczana z danych (URL + DNS/WHOIS)")
    print("• dataset_optimized.csv → Wysoka interpretowalność, optymalna wydajność")
    print("\n✅ Wszystkie cechy są reprodukowalne (URLFeatureAnalyzer + AdvancedURLAnalyzer)")

    return df_optimized

df_optimized = create_optimized_dataset_and_summarize(df_clean, keep_list, eliminate_list)

TWORZENIE ZOPTYMALIZOWANEGO ZBIORU DANYCH I PODSUMOWANIE

1. ZBIÓR DANYCH - SELEKCJA CECH (Feature Selection):
   • Oryginalna liczba cech: 110
   • Cechy po eliminacji: 47
   • Usunięte cechy: 63
   • Redukcja: 57.3%


   ✓ Zapisano do: /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/backend/data/url_corpus/quality_v4_repaired_features/dataset_optimized_train.csv
   ✓ Zapisano do: /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/backend/data/url_corpus/quality_v4_repaired_features/dataset_optimized_val.csv


   ✓ Zapisano do: /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/backend/data/url_corpus/quality_v4_repaired_features/dataset_optimized_test.csv

PORÓWNANIE ZBIORÓW DANYCH
               Zbiór danych  Liczba cech                             Typ cech Interpretowalność                  Plik
               Full (pełny)          110 Wszystkie atrybuty (URL + DNS/WHOIS)            Wysoka      dataset_full.csv
Optimized (eliminacja cech)           47 Wszystkie atrybuty (URL + DNS/WHOIS)            Wysoka dataset_optimized.csv

2. PODSUMOWANIE ANALIZY ZAPISANE:
   ✓ Lokalizacja: /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/results/url_corpus/statistical_analysis/docs/analysis_summary.md

REKOMENDACJE UŻYCIA:
• dataset_full.csv → Maksymalna informacja, liczba cech wyliczana z danych (URL + DNS/WHOIS)
• dataset_optimized.csv → Wysoka interpretowalność, optymalna wydajność

✅ Wszystkie cechy są reprodukowalne (URLFeatureAnalyzer + AdvancedURLAnalyzer

## 10. Podsumowanie Zapisanych Plików


In [10]:
from pathlib import Path


def _file_size_label(path):
    '''Return a readable file-size label.'''
    size = path.stat().st_size
    if size > 1024 ** 2:
        return f'{size / 1024 ** 2:.2f} MB'
    if size > 1024:
        return f'{size / 1024:.2f} KB'
    return f'{size} B'


def _print_artifact(path):
    '''Print the existence and size of one expected artifact.'''
    if path.exists():
        print(f'  ✅ {path} ({_file_size_label(path)})')
    else:
        print(f'  ❌ {path} (nie znaleziono)')


def _distribution_artifacts(extension):
    '''Return independently generated feature-distribution figures.'''
    root = Path(png_dir if extension == 'png' else html_dir) / 'distributions'
    return sorted(root.glob(f'*.{extension}'))


saved_files = {
    'Zbiory danych': [
        Path('../data/dataset_full.csv'),
        Path('../data/dataset_optimized.csv'),
    ],
    'Raporty CSV': [
        Path(csv_dir) / 'descriptive_statistics.csv',
        Path(csv_dir) / 'target_correlations.csv',
        Path(csv_dir) / 'feature_importance.csv',
        Path(csv_dir) / 'distribution_analysis.csv',
        Path(csv_dir) / 'missing_values_minus1.csv',
        Path(csv_dir) / 'features_to_eliminate.csv',
        Path(csv_dir) / 'features_to_keep.csv',
    ],
    'Wizualizacje PNG': [
        Path(png_dir) / 'target_correlations.png',
        Path(png_dir) / 'feature_importance.png',
        *_distribution_artifacts('png'),
        Path(png_dir) / 'missing_values_minus1.png',
        Path(png_dir) / 'correlation_heatmap.png',
    ],
    'Wizualizacje HTML': [
        Path(html_dir) / 'target_correlations.html',
        Path(html_dir) / 'feature_importance.html',
        *_distribution_artifacts('html'),
        Path(html_dir) / 'missing_values_minus1.html',
        Path(html_dir) / 'correlation_heatmap.html',
    ],
    'Dokumentacja': [Path(docs_dir) / 'analysis_summary.md'],
}

print('=' * 80)
print('PODSUMOWANIE WSZYSTKICH ZAPISANYCH PLIKÓW')
print('=' * 80)
for category, paths in saved_files.items():
    print(f'\n{category}:')
    for path in paths:
        _print_artifact(path)

png_count = len(_distribution_artifacts('png'))
html_count = len(_distribution_artifacts('html'))
print(f'\nOsobne rozkłady cech: PNG={png_count}/8, HTML={html_count}/8')
if png_count != 8 or html_count != 8:
    print('⚠️ Oczekiwano po osiem osobnych wykresów rozkładów.')

print('\n' + '=' * 80)
print(f'Analiza statystyczna: {output_dir}')
print('✅ Analiza zakończona pomyślnie.')


PODSUMOWANIE WSZYSTKICH ZAPISANYCH PLIKÓW

Zbiory danych:
  ✅ ../data/dataset_full.csv (23.91 MB)
  ✅ ../data/dataset_optimized.csv (16.30 MB)

Raporty CSV:
  ✅ /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/results/url_corpus/statistical_analysis/csv/descriptive_statistics.csv (9.03 KB)
  ✅ /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/results/url_corpus/statistical_analysis/csv/target_correlations.csv (5.25 KB)
  ✅ /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/results/url_corpus/statistical_analysis/csv/feature_importance.csv (3.66 KB)
  ✅ /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/results/url_corpus/statistical_analysis/csv/distribution_analysis.csv (5.83 KB)
  ✅ /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/results/url_corpus/statistical_analysis/csv/missing_values_minus1.csv (510 B)
  ✅ /Users/maksymiliansowula/Documents/Studia/Magisterka/PhisingDetector/results/url